# Benchmark Chunkformer on ViMD Dataset (Kaggle)

**Fixes implemented:**
- **Monkey Patch** `torchaudio.load` using `librosa` (fix codec issues).
- **Path Resolution**: Automatically joins relative paths with `DATA_DIR`.
- **Bytes Fallback**: Writes raw bytes to temp file if no physical file found.
- **Nested Output Parsing**: Correctly reads `{gender: {label: ...}}` structure.

In [ ]:
# Install dependencies
!pip install -q datasets transformers torch soundfile librosa scikit-learn chunkformer

In [ ]:
import os
import io
import torch
import torchaudio
import soundfile as sf
import librosa
import numpy as np
import pandas as pd
import tempfile
import warnings
from tqdm.auto import tqdm
from datasets import Dataset, Audio, load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from chunkformer import ChunkFormerModel

warnings.filterwarnings("ignore")

def my_torchaudio_load(filepath, **kwargs):
    try:
        wav, sr_native = librosa.load(filepath, sr=None, mono=True)
        tensor = torch.tensor(wav).float().unsqueeze(0)
        return tensor, sr_native
    except Exception as e:
        print(f"MonkeyPatch Load Error for {filepath}: {e}")
        raise e
torchaudio.load = my_torchaudio_load

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Load ViMD Dataset
DATA_DIR = "/kaggle/input/vimd-dataset"

if os.path.exists(DATA_DIR):
    print(f"Found data directory: {DATA_DIR}")
    try:
        dataset = load_dataset(DATA_DIR, split="test")
    except:
         for s in ['validation', 'valid', 'train']:
             try: 
                 dataset = load_dataset(DATA_DIR, split=s)
                 break
             except: continue
else:
    print(f"Fallback to Hub.")
    dataset = load_dataset("nguyendv02/ViMD_Dataset", split="test")

dataset = dataset.cast_column("audio", Audio(decode=False))
print(f"Dataset Size: {len(dataset)}")

In [ ]:
# Load Model
model_id = "khanhld/chunkformer-gender-emotion-dialect-age-classification"
model = ChunkFormerModel.from_pretrained(model_id)
model.to(device)
model.eval()
print("Model loaded.")

In [ ]:
# Benchmark Loop
preds = {'gender': [], 'dialect': []}
labels = {'gender': [], 'dialect': []}

print("Running inference...")
pbar = tqdm(dataset)
skip_count, valid_count = 0, 0

for i, item in enumerate(pbar):
    is_temp = False
    path = None
    try:
        audio_src = item.get('audio', {})
        if isinstance(audio_src, dict):
            path = audio_src.get('path') or audio_src.get('file')
        elif isinstance(audio_src, str): path = audio_src
            
        # 1. Path Resolution (Absolute check)
        if path and not os.path.exists(path):
             for base in [DATA_DIR, os.path.join(DATA_DIR, 'test'), os.path.join(DATA_DIR, 'wav')]:
                 p_cand = os.path.join(base, path)
                 if os.path.exists(p_cand):
                     path = p_cand
                     break
        
        # 2. Bytes Fallback
        if (not path or not os.path.exists(path)) and audio_src.get('bytes'):
             with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                 tmp.write(audio_src['bytes'])
                 path = tmp.name
                 is_temp = True
        
        if not path or not os.path.exists(path):
             skip_count += 1
             continue

        # 3. Predict
        with torch.no_grad():
             res = model.classify_audio(audio_path=path, chunk_size=-1, left_context_size=-1, right_context_size=-1)
        
        # 4. Parse
        g_pred, d_pred = 'Unknown', 'Unknown'
        if isinstance(res, dict):
            if 'gender' in res: g_pred = 'Female' if 'female' in res['gender'].get('label','').lower() else 'Male'
            if 'dialect' in res:
                lbl = res['dialect'].get('label','').lower()
                if 'north' in lbl or 'minority' in lbl: d_pred = 'North'
                elif 'central' in lbl: d_pred = 'Central'
                elif 'south' in lbl: d_pred = 'South'
                else: d_pred = lbl
        
        preds['gender'].append(g_pred)
        preds['dialect'].append(d_pred)
        
        labels['gender'].append(item.get('gender', item.get('sex', -1)))
        labels['dialect'].append(item.get('region', item.get('dialect', 'Unknown')))
        valid_count += 1

    except Exception as e:
        if i < 5: print(f"Error idx {i}: {e}")
        continue
    finally:
        if is_temp and path and os.path.exists(path): os.remove(path)
        
print(f"Completed. Valid: {valid_count}, Skipped: {skip_count}")

In [ ]:
# Evaluation
if len(labels['dialect']) > 0:
    it_reg = {0: 'North', 1: 'Central', 2: 'South'}
    f_gt_d = [it_reg.get(l, str(l)) if isinstance(l, int) else str(l) for l in labels['dialect']]
    f_pred_d = preds['dialect']
    
    f_gt_g = ['Male' if l == 0 else 'Female' if l == 1 else str(l) for l in labels['gender']]
    f_pred_g = preds['gender']

    print("\n[DIALECT] Report")
    print(classification_report(f_gt_d, f_pred_d))
    print("\n[GENDER] Report")
    print(classification_report(f_gt_g, f_pred_g))
else:
    print("No data to evaluate.")